# 17 FS3 Staged Ablation Evaluation

This notebook is the combined evaluation and conclusion layer for the upgraded `FS3` staged ablation workflow across `LEAR` and `XGBoost`.

Active scope:
- `Stage A` synthesis across both models
- `Layer 1` synthesis across both models
- `Layer 2` only for compatible subgroup runs that were actually executed
- lightweight model-specific diagnostics as supportive evidence

Excluded from the main thesis-grade path by default:
- legacy `FS1` / `FS2` smoke-test ablation contexts
- old FS3 combo results that predate the fixed taxonomy or the current scheme hashes
- any saved aggregate whose compatibility checks no longer pass


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.external_features import build_external_family_catalog, load_external_feature_store
from hourly_da.core.methodology import (
    STARTER_ENDOGENOUS_FEATURE_NOTE,
    feature_stage_policy_frame,
    model_status_frame,
    shortlisting_policy_frame,
)
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.core.tuning import build_tuning_placeholder, tuning_cadence_frame, tuning_snippet_frame
from hourly_da.models import naive_model_names
from hourly_da.notebook_support import estimate_run_duration_seconds, format_duration, load_selected_case_weeks

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root


def latest_run_or_none(run_label: str) -> Path | None:
    try:
        return find_latest_run(output_root, run_label)
    except FileNotFoundError:
        return None


def ensure_required_modules(module_names: list[str], install_command: str | None = None) -> None:
    missing = [module_name for module_name in module_names if importlib.util.find_spec(module_name) is None]
    if not missing:
        return

    message_lines = [
        "Missing required package(s) in the active notebook interpreter: " + ", ".join(missing),
        f"Active interpreter: {sys.executable}",
    ]
    if install_command:
        message_lines.append(f"Install command: {install_command}")
    raise RuntimeError("\n".join(message_lines))


def run_command_with_live_output(command: list[str]) -> None:
    print("Running command:")
    print(" ".join(str(part) for part in command))
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    output_tail: list[str] = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        output_tail.append(line)
        if len(output_tail) > 40:
            output_tail.pop(0)

    return_code = process.wait()
    if return_code != 0:
        tail_text = "".join(output_tail).strip()
        message = f"Command failed with exit code {return_code}."
        if tail_text:
            message += "\nLast output:\n" + tail_text
        raise RuntimeError(message)


## Purpose

What this notebook does:
- combines the compatible `FS3` staged-ablation results from notebooks `15` and `16`
- keeps `Stage A`, `Layer 1`, and `Layer 2` separate instead of averaging across unlike exercises
- summarizes what both models jointly suggest in plain language

How to read it:
- `validation` MAE deltas are the primary ranking evidence
- `test` and diagnostics are secondary consistency checks
- positive ablation delta means the child got worse after removal, so the removed block was helping in that parent context


In [ ]:
from IPython.display import Markdown

from hourly_da.core.ablation_blocks import (
    SCHEME_NAME_LAYER_1,
    SCHEME_NAME_LAYER_2,
    SCHEME_NAME_STAGE_A,
    supported_layer2_target_blocks,
)
from hourly_da.notebook_support import (
    ablation_effect_label,
    annotate_ablation_metric_slice,
    apply_notebook_display_defaults,
    compute_fs3_parent_block_diagnostics,
    load_fs3_ablation_report_dataset,
    select_feature_family_metric_slice,
    summarize_ablation_cross_model,
)

apply_notebook_display_defaults()

RUN_DIAGNOSTICS = False
PRIMARY_SPLIT = "validation"
PRIMARY_METRIC = "mae"
VISIBLE_REPORTING_LEVELS = ("d_only", "stitched_all_horizon")

_combined_store = load_external_feature_store(config)
_combined_layer2_targets = supported_layer2_target_blocks(_combined_store.experiment_map())
FS3_COMBINED_PARENT_SPECS = [
    {
        "parent_run_label": "lear_fs3_combo_promoted_benchmark",
        "model_family": "lear",
        "model_label": "LEAR",
    },
    {
        "parent_run_label": "xgboost_fs3_combo_promoted_benchmark",
        "model_family": "xgboost",
        "model_label": "XGBoost",
    },
]
FS3_COMBINED_SCHEME_REQUESTS = [
    {"scheme_name": SCHEME_NAME_STAGE_A, "target_block": None},
    {"scheme_name": SCHEME_NAME_LAYER_1, "target_block": None},
]
FS3_COMBINED_SCHEME_REQUESTS.extend(
    {"scheme_name": SCHEME_NAME_LAYER_2, "target_block": target_block}
    for target_block in _combined_layer2_targets
)

FS3_COMBINED_REPORT = load_fs3_ablation_report_dataset(
    output_root,
    config,
    parent_specs=FS3_COMBINED_PARENT_SPECS,
    scheme_requests=FS3_COMBINED_SCHEME_REQUESTS,
)
FS3_COMBINED_AVAILABILITY = FS3_COMBINED_REPORT["availability"]
FS3_COMBINED_BUNDLES = FS3_COMBINED_REPORT["bundles"]
FS3_COMBINED_SUMMARY = FS3_COMBINED_REPORT["summary"]
FS3_COMBINED_BY_ORIGIN = FS3_COMBINED_REPORT["by_origin"]


## Compatibility Gate


In [ ]:
availability_view = FS3_COMBINED_AVAILABILITY[
    [
        "model_label",
        "scheme_name",
        "layer_name",
        "target_block",
        "availability_status",
        "latest_timestamp_label",
        "status_note",
    ]
].rename(
    columns={
        "model_label": "Model",
        "scheme_name": "Scheme",
        "layer_name": "Layer",
        "target_block": "Layer 2 target",
        "availability_status": "Status",
        "latest_timestamp_label": "Latest timestamp",
        "status_note": "Status note",
    }
)
display(availability_view.style.hide(axis="index"))

display(
    Markdown(
        "Active synthesis scope: only compatible FS3 combo staged-ablation bundles are included below. "
        "Legacy FS1/FS2 smoke contexts and old incompatible FS3 bundles are kept out of the thesis-grade path by default."
    )
)


## Stage A Across Both Models

This section answers the broad orientation question:
- how much information comes from endogenous history,
- how much comes from calendar effects,
- and how much comes from the total exogenous bundle?

The goal is not fine-grained attribution yet. The goal is to establish where the broad predictive signal seems to live.


In [ ]:
scheme_summary = FS3_COMBINED_SUMMARY[
    FS3_COMBINED_SUMMARY["scheme_name"].astype(str) == 'stage_a_top_level'
].copy()

if scheme_summary.empty:
    print("No compatible saved results are available for this layer yet.")
else:
    for reporting_level in VISIBLE_REPORTING_LEVELS:
        metric_slice = select_feature_family_metric_slice(
            scheme_summary,
            split=PRIMARY_SPLIT,
            reporting_level=reporting_level,
            metric=PRIMARY_METRIC,
        )
        display(Markdown(f"### Validation / {reporting_level.replace('_', ' ').title()}"))
        if metric_slice.empty:
            print("No rows are available for this reporting lens.")
            continue

        annotated = annotate_ablation_metric_slice(metric_slice)
        model_view = annotated[
            [
                "model_family",
                "feature_family",
                "parent_value",
                "child_value",
                "delta",
                "relative_delta",
                "effect_label",
            ]
        ].rename(
            columns={
                "model_family": "Model family",
                "feature_family": "Block",
                "parent_value": "Parent value",
                "child_value": "Child value",
                "delta": "Delta",
                "relative_delta": "Relative delta",
                "effect_label": "Interpretation",
            }
        )
        display(
            model_view.style
            .format(
                {
                    "Parent value": "{:.4f}",
                    "Child value": "{:.4f}",
                    "Delta": "{:+.4f}",
                    "Relative delta": "{:+.3%}",
                }
            )
            .hide(axis="index")
        )

        combined = summarize_ablation_cross_model(metric_slice)
        display(
            combined.rename(
                columns={
                    "feature_family": "Block",
                    "models_available": "Models available",
                    "model_count": "Model count",
                    "mean_delta": "Mean delta",
                    "mean_relative_delta": "Mean relative delta",
                    "combined_label": "Combined classification",
                }
            ).style
            .format(
                {
                    "Mean delta": "{:+.4f}",
                    "Mean relative delta": "{:+.3%}",
                }
            )
            .hide(axis="index")
        )


## Layer 1 Across Both Models

This is the main thesis-grade attribution layer.

What to look for:
- blocks that help both models under the same reporting lens
- blocks that look model-dependent or unstable
- blocks whose effect is near-zero and may not justify their complexity

Important caution:
- a block can look weak on the stitched horizon even when it matters strongly for `D only`
- that is a reporting-lens issue, not necessarily a feature-quality issue


In [ ]:
scheme_summary = FS3_COMBINED_SUMMARY[
    FS3_COMBINED_SUMMARY["scheme_name"].astype(str) == 'layer1_mutually_exclusive'
].copy()

if scheme_summary.empty:
    print("No compatible saved results are available for this layer yet.")
else:
    for reporting_level in VISIBLE_REPORTING_LEVELS:
        metric_slice = select_feature_family_metric_slice(
            scheme_summary,
            split=PRIMARY_SPLIT,
            reporting_level=reporting_level,
            metric=PRIMARY_METRIC,
        )
        display(Markdown(f"### Validation / {reporting_level.replace('_', ' ').title()}"))
        if metric_slice.empty:
            print("No rows are available for this reporting lens.")
            continue

        annotated = annotate_ablation_metric_slice(metric_slice)
        model_view = annotated[
            [
                "model_family",
                "feature_family",
                "parent_value",
                "child_value",
                "delta",
                "relative_delta",
                "effect_label",
            ]
        ].rename(
            columns={
                "model_family": "Model family",
                "feature_family": "Block",
                "parent_value": "Parent value",
                "child_value": "Child value",
                "delta": "Delta",
                "relative_delta": "Relative delta",
                "effect_label": "Interpretation",
            }
        )
        display(
            model_view.style
            .format(
                {
                    "Parent value": "{:.4f}",
                    "Child value": "{:.4f}",
                    "Delta": "{:+.4f}",
                    "Relative delta": "{:+.3%}",
                }
            )
            .hide(axis="index")
        )

        combined = summarize_ablation_cross_model(metric_slice)
        display(
            combined.rename(
                columns={
                    "feature_family": "Block",
                    "models_available": "Models available",
                    "model_count": "Model count",
                    "mean_delta": "Mean delta",
                    "mean_relative_delta": "Mean relative delta",
                    "combined_label": "Combined classification",
                }
            ).style
            .format(
                {
                    "Mean delta": "{:+.4f}",
                    "Mean relative delta": "{:+.3%}",
                }
            )
            .hide(axis="index")
        )


## Layer 2 Follow-Up Results

Layer 2 is only shown for subgroup bundles that were actually run and still pass compatibility checks.

These results answer a narrower question:
- within one chosen Layer 1 block, which subgroup still adds marginal value while the rest of the full parent bundle stays fixed?


In [ ]:
available_layer2 = FS3_COMBINED_SUMMARY[
    FS3_COMBINED_SUMMARY["scheme_name"].astype(str) == "layer2_subgroups"
].copy()

if available_layer2.empty:
    print("No compatible Layer 2 subgroup results are available yet.")
else:
    for target_block, group in available_layer2.groupby("target_block", dropna=False):
        display(Markdown(f"### {str(target_block).replace('_', ' ').title()}"))
        for reporting_level in VISIBLE_REPORTING_LEVELS:
            metric_slice = select_feature_family_metric_slice(
                group,
                split=PRIMARY_SPLIT,
                reporting_level=reporting_level,
                metric=PRIMARY_METRIC,
            )
            if metric_slice.empty:
                continue
            annotated = annotate_ablation_metric_slice(metric_slice)
            display(Markdown(f"#### {reporting_level.replace('_', ' ').title()}"))
            display(
                annotated[
                    [
                        "model_family",
                        "feature_family",
                        "delta",
                        "relative_delta",
                        "effect_label",
                    ]
                ]
                .rename(
                    columns={
                        "model_family": "Model family",
                        "feature_family": "Subgroup",
                        "delta": "Delta",
                        "relative_delta": "Relative delta",
                        "effect_label": "Interpretation",
                    }
                )
                .style
                .format({"Delta": "{:+.4f}", "Relative delta": "{:+.3%}"})
                .hide(axis="index")
            )


## Supportive Diagnostics

These diagnostics are lightweight and model-specific.

How to interpret them:
- coefficient activity or tree gain can support an ablation story
- but they should never replace the out-of-sample ablation result
- disagreement between diagnostics and ablation is possible, especially with correlated blocks


In [ ]:
if not RUN_DIAGNOSTICS:
    print("Diagnostics are disabled for this notebook.")
else:
    for parent_spec in FS3_COMBINED_PARENT_SPECS:
        display(Markdown(f"### {parent_spec['model_label']}"))
        diagnostics = compute_fs3_parent_block_diagnostics(
            config,
            parent_run_label=str(parent_spec["parent_run_label"]),
            model_family=str(parent_spec["model_family"]),
            split_name="validation",
        )
        display(diagnostics["reference"])
        for diagnostic_block in diagnostics["tables"]:
            display(
                Markdown(
                    f"#### {diagnostic_block['branch_name'].replace('_', ' ').title()} / "
                    f"{diagnostic_block['diagnostic_type'].replace('_', ' ').title()}"
                )
            )
            table = diagnostic_block["table"]
            if diagnostic_block["diagnostic_type"] == "lear_coefficients":
                display(
                    table.style
                    .format({"sum_abs_standardized_coefficient": "{:.4f}"})
                    .hide(axis="index")
                )
            else:
                display(
                    table.style
                    .format({"total_gain": "{:.4f}", "total_split_count": "{:.0f}"})
                    .hide(axis="index")
                )


## Final Conclusion

How to summarize the combined evidence:
- blocks that are strongly positive in both models are the most trustworthy keepers
- blocks that are strongly negative in both models are the strongest removal candidates
- blocks with mixed signs or near-zero deltas need cautious interpretation
- `Layer 2` should only be used as a targeted follow-up, not as a replacement for the broader `Stage A` and `Layer 1` view

Remaining limitations:
- this is still rolling-origin evidence, not fold CV
- marginal block value is conditional on the chosen parent tuning and parent feature bundle
- diagnostics are based on one representative validation fit, not a full multi-origin native-importance study
